In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
df = sns.load_dataset('titanic')
df.head(3)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


In [3]:
df = df[['survived', 'pclass','sex','age','sibsp','parch','fare','embarked']]
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [4]:
from sklearn.model_selection import train_test_split

In [5]:
X_train,X_test, y_train, y_test = train_test_split(df.drop('survived', axis=1),df['survived'],test_size=0.2, random_state=42)

In [6]:
X_train.head(2)

,pclass,sex,age,sibsp,parch,fare,embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [7]:
X_test.head(2)

,pclass,sex,age,sibsp,parch,fare,embarked
709,3,male,NaN,1,1,15.2458,C
439,2,male,31.0,0,0,10.5000,S


In [8]:
y_train.head()

331    0
733    0
382    0
704    0
813    0
Name: survived, dtype: int64

In [9]:
df.isnull().sum()

survived      0
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64

---
# Simple imputer

In [10]:
from sklearn.impute import SimpleImputer

In [11]:
si = SimpleImputer()
si_embarked = SimpleImputer(strategy='most_frequent')

In [12]:
X_train_age = si.fit_transform(X_train[['age']])
X_train_embarked = si_embarked.fit_transform(X_train[['embarked']])

In [13]:
X_test_age = si.transform(X_test[['age']])
X_test_embarked = si_embarked.transform(X_test[['embarked']])

---
# One Hot Encoding

In [14]:
from sklearn.preprocessing import OneHotEncoder

In [15]:
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore',dtype=np.int8)
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore',dtype=np.int8)

In [16]:
X_train_sex = ohe_sex.fit_transform(X_train[['sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

In [17]:
# TEST (IMPORTANT)
X_test_sex = ohe_sex.transform(X_test[['sex']])
X_test_embarked = ohe_embarked.transform(X_test[['embarked']])

C:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [18]:
X_train_embarked

array([[0, 0, 1],
       [0, 0, 1],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [0, 0, 1],
       [0, 0, 1]], dtype=int8)

In [19]:
df['embarked'].value_counts()

embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [20]:
X_train_rem = X_train.drop(columns=['sex','age','embarked'])
X_test_rem = X_test.drop(columns=['sex','age','embarked'])

In [21]:
X_train_transformed = np.concatenate((X_train_rem,X_train_sex,X_train_age,X_train_embarked),axis=1)
X_test_transformed = np.concatenate((X_test_rem,X_test_sex,X_test_age,X_test_embarked),axis=1)

In [22]:
X_test_transformed.shape

(179, 10)

In [23]:
pd.DataFrame(X_train_transformed)

,0,1,2,3,4,5,6,7,8,9
0,1.0,0.0,0.0,28.5000,0.0,1.0,45.500000,0.0,0.0,1.0
1,2.0,0.0,0.0,13.0000,0.0,1.0,23.000000,0.0,0.0,1.0
2,3.0,0.0,0.0,7.9250,0.0,1.0,32.000000,0.0,0.0,1.0
3,3.0,1.0,0.0,7.8542,0.0,1.0,26.000000,0.0,0.0,1.0
4,3.0,4.0,2.0,31.2750,1.0,0.0,6.000000,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
707,3.0,0.0,0.0,7.6500,1.0,0.0,21.000000,0.0,0.0,1.0
708,1.0,0.0,0.0,31.0000,0.0,1.0,29.498846,0.0,0.0,1.0
709,3.0,2.0,0.0,14.1083,0.0,1.0,41.000000,0.0,0.0,1.0
710,1.0,1.0,2.0,120.0000,1.0,0.0,14.000000,0.0,0.0,1.0


In [24]:
from sklearn.tree import DecisionTreeClassifier

In [25]:
clf = DecisionTreeClassifier()

In [26]:
model = clf.fit(X_train_transformed,y_train)
model

DecisionTreeClassifier()

In [27]:
y_pred = clf.predict(X_test_transformed)
y_pred

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 1])

In [28]:
from sklearn.metrics import accuracy_score

In [29]:
accuracy_score(y_test,y_pred)

0.7821229050279329

In [30]:
import pickle

In [31]:
pickle.dump(ohe_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf,open('models/clf.pkl','wb'))
